In [ ]:
import torch
from torch.utils.data import DataLoader, ConcatDataset
import torch.nn as nn
import os
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torchinfo import summary
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt 
import numpy as np 
from sklearn.metrics import confusion_matrix
from features import getFeatures
from utils.timeseriesloader import TimeSeriesDataset
from load_data import load_data
from models.AlphaModel import AlphaModel

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
print('The model is running on:', DEVICE) 

NUM_FEATURES = 10
NUM_CLASSES = 4
BATCH_SIZE = 32
EPOCHS = 50

In [ ]:
train_data, val_data, test_data = load_data(if_test=False) 

print("Train data: ", len(train_data))
print("Test data: ", len(test_data))
print("Val data: ", len(val_data))

conc_train = ConcatDataset(train_data)
conc_test = ConcatDataset(test_data)
conc_val = ConcatDataset(val_data)

train_loader = DataLoader(conc_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)
test_loader = DataLoader(conc_test, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)
val_loader = DataLoader(conc_val, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)

print("Data", len(train_loader), len(test_loader), len(val_loader))

# Model

In [ ]:
model = AlphaModel(num_inputs=NUM_FEATURES).to(DEVICE)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter('alpha_runs/runs_{}'.format(timestamp))
model_directory = os.path.join('alpha_model', 'model_{}'.format(timestamp))
    
print(summary(model, input_size=(BATCH_SIZE, 200, NUM_FEATURES)))

continuous_loss_fn = nn.L1Loss(reduction='none')
best_val_loss = float("inf")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=2e-6)
scheduler = ReduceLROnPlateau(optimizer, factor=0.1, patience=3)

# Training

In [ ]:
def train_one_epoch(model, optimizer, dataloader):
    model.train()
    running_loss = 0
    runs = 0

    for inputs, alpha_labels,_,_ in dataloader:

        inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)
        mask = (alpha_labels != LABEL_PADDING_VALUE).float()

        outputs = model(inputs)
        outputs = outputs.squeeze(-1)
        total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()
                
        optimizer.zero_grad()
        total_loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        
        running_loss += total_loss.item()
        runs += 1

        progress_bar.update()

    return running_loss/runs

def evaluate_model(model, dataloader):
    model.eval()
    
    running_val_total = 0.0
    val_runs = 0

    with torch.no_grad():
        for inputs, alpha_labels,_,_ in dataloader:
            
            inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)
            mask = (alpha_labels != LABEL_PADDING_VALUE).float()
            
            outputs = model(inputs)  
            outputs = outputs.squeeze(-1)
            total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()            
            running_val_total += total_loss.item()
            val_runs += 1
    
    return running_val_total / val_runs

In [ ]:
os.makedirs(model_directory, exist_ok=True)

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch + 1))

    progress_bar = tqdm(total=len(train_loader), desc='Training', position=0)

    avg_training_loss = train_one_epoch(model, optimizer, train_loader)
    val_total_loss  = evaluate_model(model, val_loader)
    
    print(f'Training LOSS: Alpha {avg_training_loss}\n'
          f'Validation LOSS: Alpha {val_total_loss} \n')
    
    writer.add_scalars('Losses', {
        'Training Alpha Loss': avg_training_loss,
        'Validation Alpha Loss': val_total_loss,
        }, epoch + 1)

    writer.flush()
    
    if val_total_loss < best_val_loss:
        best_val_loss = val_total_loss
        best_model_path = os.path.join(model_directory, f'model_{epoch + 1}')
        torch.save(model.state_dict(), best_model_path)

    scheduler.step(val_total_loss)
    
progress_bar.close()
writer.close()

In [ ]:
print("Best Validation Loss:", best_val_loss)
print("Best Model Path", best_model_path)

# Testing

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.eval()

running_test_total = 0.0
test_runs = 0.0

predictions = []
ground_truth = []

progress_bar = tqdm(total=len(test_loader), desc='Testing', position=0)

with torch.no_grad():
    for inputs, alpha_labels,_,_ in test_loader:
        
        inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)

        mask = (alpha_labels != LABEL_PADDING_VALUE).float()
        outputs = model(inputs).squeeze(-1)
        total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()
        
        running_test_total += total_loss.item()
        test_runs += 1

        predictions.extend(outputs.cpu().numpy())
        ground_truth.extend(alpha_labels.cpu().numpy())
        progress_bar.update()


# Calculate average losses
avg_test_loss = running_test_total / test_runs
print(f'Average test loss: {avg_test_loss}')
progress_bar.close()

# Visualise Predictions

In [ ]:
INDEX = 53

padding_starts = (ground_truth[INDEX] == LABEL_PADDING_VALUE).argmax() 

if padding_starts == 0:
    padding_starts = 200

pred_alpha = predictions[INDEX][:padding_starts]
true_alpha = ground_truth[INDEX][:padding_starts]

plt.scatter([i for i in range(len(pred_alpha))], pred_alpha, color="red")
plt.scatter([i for i in range(len(true_alpha))], true_alpha, color="blue")
plt.title("Alpha")
plt.show()